#  Data Cleaning

In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

orders = pd.read_csv(RAW / "olist_orders_dataset.csv")
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")
products = pd.read_csv(RAW / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv")
translation = pd.read_csv(RAW / "product_category_name_translation.csv")

raw_counts = {
    "orders": len(orders),
    "order_items": len(items),
    "payments": len(payments),
    "customers": len(customers),
    "products": len(products),
    "sellers": len(sellers),
}

for name, n in raw_counts.items():
    print(f"{name:12} raw rows: {n:>8,}")

orders       raw rows:   99,441
order_items  raw rows:  112,650
payments     raw rows:  103,886
customers    raw rows:   99,441
products     raw rows:   32,951
sellers      raw rows:    3,095


#   Convert Date Columns to Datetime

In [2]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

# 1. Count nulls before conversion
nulls_before = orders[date_cols].isnull().sum()

# 2. Convert text to datetime
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

# 3. Count nulls after conversion
nulls_after = orders[date_cols].isnull().sum()

# 4. Verify
print("Data types after conversion:")
print(orders[date_cols].dtypes)

print("\nNulls before vs after (should be identical):")
print(pd.DataFrame({"before": nulls_before, "after": nulls_after}))

print("\nRows in orders:", len(orders), "| raw:", raw_counts["orders"])

Data types after conversion:
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

Nulls before vs after (should be identical):
                               before  after
order_purchase_timestamp            0      0
order_approved_at                 160    160
order_delivered_carrier_date     1783   1783
order_delivered_customer_date    2965   2965
order_estimated_delivery_date       0      0

Rows in orders: 99441 | raw: 99441


** All five date columns were converted from text to datetime. Null counts are
- unchanged, so no value failed to convert and no rows were added or removed.
- Existing nulls (undelivered orders) were deliberately kept.

# : Clean Product Categories

In [3]:
# Start from the raw file every time, so this cell can be re-run safely
products = pd.read_csv(RAW / "olist_products_dataset.csv")

missing_before = products["product_category_name"].isnull().sum()
products["product_category_name"] = products["product_category_name"].fillna("unknown")

products = products.merge(translation, on="product_category_name", how="left")

# Manual translation for the 2 categories missing from the translation table
manual_translation = {
    "portateis_cozinha_e_preparadores_de_alimentos": "portable_kitchen_and_food_preparers",
    "pc_gamer": "pc_gamer",
}
no_english = products["product_category_name_english"].isnull()
products.loc[no_english, "product_category_name_english"] = (
    products.loc[no_english, "product_category_name"].map(manual_translation)
)

# Anything still empty (the "unknown" label) keeps its own name
products["product_category_name_english"] = products["product_category_name_english"].fillna(
    products["product_category_name"]
)

print("Missing categories labelled unknown :", missing_before)
print("Rows in products                    :", len(products), "| raw:", raw_counts["products"])
print("Unique category names (English)     :", products["product_category_name_english"].nunique())
print("Columns:", list(products.columns))

Missing categories labelled unknown : 610
Rows in products                    : 32951 | raw: 32951
Unique category names (English)     : 74
Columns: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english']


### Check: Category Columns After Cleaning

In [4]:
print("Missing in product_category_name         :", products["product_category_name"].isnull().sum())
print("Missing in product_category_name_english :", products["product_category_name_english"].isnull().sum())
print("Products labelled unknown                :", (products["product_category_name"] == "unknown").sum())

print("\nManually translated categories:")
display(
    products.loc[
        products["product_category_name"].isin(
            ["portateis_cozinha_e_preparadores_de_alimentos", "pc_gamer"]
        ),
        ["product_category_name", "product_category_name_english"],
    ].drop_duplicates()
)

Missing in product_category_name         : 0
Missing in product_category_name_english : 0
Products labelled unknown                : 610

Manually translated categories:


,product_category_name,product_category_name_english
1628,pc_gamer,pc_gamer
5821,portateis_cozinha_e_preparadores_de_alimentos,portable_kitchen_and_food_preparers


## Step 3: Validate and Save Cleaned Tables

In [5]:
clean_counts = {
    "orders": len(orders),
    "order_items": len(items),
    "payments": len(payments),
    "customers": len(customers),
    "products": len(products),
    "sellers": len(sellers),
}

check = pd.DataFrame({"raw": raw_counts, "clean": clean_counts})
check["match"] = check["raw"] == check["clean"]
display(check)
assert check["match"].all(), "Row counts changed during cleaning"

assert orders["order_id"].is_unique, "Duplicate order_id in orders"
assert products["product_id"].is_unique, "Duplicate product_id in products"
assert sellers["seller_id"].is_unique, "Duplicate seller_id in sellers"
assert customers["customer_id"].is_unique, "Duplicate customer_id in customers"
print("All validation checks passed")

orders.to_csv(PROCESSED / "orders_clean.csv", index=False)
items.to_csv(PROCESSED / "order_items_clean.csv", index=False)
payments.to_csv(PROCESSED / "payments_clean.csv", index=False)
customers.to_csv(PROCESSED / "customers_clean.csv", index=False)
products.to_csv(PROCESSED / "products_clean.csv", index=False)
sellers.to_csv(PROCESSED / "sellers_clean.csv", index=False)


test = pd.read_csv(PROCESSED / "orders_clean.csv", parse_dates=[
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
])
print("\nSaved files:", sorted(p.name for p in PROCESSED.glob("*_clean.csv")))
print("orders_clean.csv rows:", len(test))
print(test["order_purchase_timestamp"].dtype)

,raw,clean,match
orders,99441,99441,True
order_items,112650,112650,True
payments,103886,103886,True
customers,99441,99441,True
products,32951,32951,True
sellers,3095,3095,True


All validation checks passed

Saved files: ['customers_clean.csv', 'order_items_clean.csv', 'orders_clean.csv', 'payments_clean.csv', 'products_clean.csv', 'sellers_clean.csv']
orders_clean.csv rows: 99441
datetime64[ns]


## Cleaning Summary

- Converted 5 date columns in orders from text to datetime; nulls unchanged.
- Labelled 610 products with a missing category as `unknown`, added English category
  names, and translated 2 missing categories manually.
- Row counts match the raw data for all 6 tables and key columns are unique.
- Saved cleaned tables to data/processed/.